# Step 12: Deploy the Model to an Endpoint

**SageMaker Unified Studio Component**: Inference Endpoints

Deploys the trained model as a real-time SageMaker endpoint. Loads the registered model from MLflow, packages it as `model.tar.gz`, and deploys via SageMaker `SKLearnModel`.

In [ ]:
# Ensure sagemaker-mlflow plugin is installed for ARN-based tracking
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "sagemaker-mlflow", "--quiet", "--upgrade"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"pip install failed: {result.stderr}")
else:
    print("sagemaker-mlflow installed/upgraded")

In [ ]:
# Parameters (injected by workflow via papermill)
mlflow_tracking_uri = "arn:aws:sagemaker:eu-west-1:146103651929:mlflow-tracking-server/machine-overheat-mlflow"
endpoint_name = "machine-overheat-endpoint"

In [ ]:
import sagemaker
import boto3
import json
import os
import re
import mlflow
import mlflow.sklearn
import joblib
import tarfile
from sagemaker.sklearn import SKLearnModel
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
from botocore.exceptions import ClientError

account_id = boto3.client('sts').get_caller_identity()['Account']
bucket_name = f'sagemaker-unified-overheat-demo-{account_id}'
print(f"Using bucket: {bucket_name}")

role = sagemaker.get_execution_role()
print(f"Execution role: {role}")

## Setup MLflow

In [ ]:
try:
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment("machine-overheat")
    print(f"MLflow tracking URI: {mlflow_tracking_uri}")
except Exception as e:
    print(f"ARN-based tracking failed: {e}")
    match = re.search(r'arn:aws:sagemaker:([^:]+):[^:]+:mlflow-(?:app|tracking-server)/(.+)', mlflow_tracking_uri)
    if match:
        region, app_id = match.group(1), match.group(2)
        tracking_url = f"https://{app_id}.mlflow.sagemaker.{region}.app.aws"
        os.environ['MLFLOW_TRACKING_URI'] = tracking_url
        os.environ['MLFLOW_TRACKING_AWS_SIGV4'] = 'true'
        os.environ['AWS_DEFAULT_REGION'] = region
        mlflow.set_tracking_uri(tracking_url)
        mlflow.set_experiment("machine-overheat")
        print(f"MLflow set with URL fallback: {tracking_url}")
    else:
        raise

## Load Model from MLflow and Package for SageMaker

In [ ]:
# Load registered model from MLflow
model_name = "machine-overheat-model"
model = mlflow.sklearn.load_model(f"models:/{model_name}/latest")
print(f"✓ Loaded model: {model_name} (latest version)")

# Write inference.py to disk — SKLearnModel needs the entry_point file locally
inference_code = '''import joblib, json, numpy as np

def model_fn(model_dir):
    return joblib.load(f"{model_dir}/model.pkl")

def input_fn(request_body, content_type):
    if content_type == 'application/json':
        data = json.loads(request_body)
        temp = data['temperature']
        room_temp = data['room_temp']
        temp_diff = temp - room_temp
        return np.array([[temp, temp_diff]])
    raise ValueError(f"Unsupported content type: {content_type}")

def predict_fn(input_data, model):
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]
    return {'prediction': int(prediction), 'probability': float(probability)}

def output_fn(prediction, accept):
    return json.dumps(prediction), accept
'''

# Package model + inference script into model.tar.gz
joblib.dump(model, '/tmp/model.pkl')
with open('/tmp/inference.py', 'w') as f:
    f.write(inference_code)
with open('inference.py', 'w') as f:
    f.write(inference_code)

with tarfile.open('/tmp/model.tar.gz', 'w:gz') as tar:
    tar.add('/tmp/model.pkl', arcname='model.pkl')
    tar.add('/tmp/inference.py', arcname='inference.py')

# Upload to S3
s3 = boto3.client('s3')
s3.upload_file('/tmp/model.tar.gz', bucket_name, 'models/logistic_regression/model.tar.gz')
print(f"✓ model.tar.gz packaged from MLflow and uploaded to s3://{bucket_name}/models/logistic_regression/model.tar.gz")

## Deploy Endpoint

In [ ]:
model_data = f's3://{bucket_name}/models/logistic_regression/model.tar.gz'

# Delete existing endpoint if present (redeploy-safe)
sm_client = boto3.client('sagemaker')
try:
    sm_client.describe_endpoint(EndpointName=endpoint_name)
    print(f"Endpoint '{endpoint_name}' exists — deleting for fresh deploy...")
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    import time
    while True:
        try:
            sm_client.describe_endpoint(EndpointName=endpoint_name)
            time.sleep(10)
        except ClientError:
            break
    print("Old endpoint deleted.")
except ClientError:
    print(f"No existing endpoint '{endpoint_name}' — creating new.")

sklearn_model = SKLearnModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    framework_version='1.2-1',
    py_version='py3'
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name=endpoint_name
)
print(f"✓ Endpoint deployed: {predictor.endpoint_name}")

## Smoke Test

In [ ]:
predictor = Predictor(
    endpoint_name=endpoint_name,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

# Normal temperature — should predict 0
result_normal = predictor.predict({'temperature': 72, 'room_temp': 25})
print(f"Normal (72°C):  prediction={result_normal['prediction']}, prob={result_normal['probability']:.4f}")
assert result_normal['prediction'] == 0, "Smoke test FAILED: normal temp predicted as overheat"

# Overheat temperature — should predict 1
result_hot = predictor.predict({'temperature': 85, 'room_temp': 25})
print(f"Overheat (85°C): prediction={result_hot['prediction']}, prob={result_hot['probability']:.4f}")
assert result_hot['prediction'] == 1, "Smoke test FAILED: overheat temp predicted as normal"

print("\n\u2713 Smoke tests passed — endpoint is ready")